In [1]:
import pandas as pd
import numpy as np
import json

In [7]:
# Load preprocessed datasets
users_df = pd.read_csv("/content/users_clean.csv")
events_df = pd.read_csv("/content/events_clean_final.csv")

with open("/content/final_rule_engine_output.json", "r") as f:
    users_json = json.load(f)


In [8]:
print(events_df.shape)
events_df.head()


(13, 7)


,event_id,user_id,event_type,event_intensity,event_recency_days,description,severity_score
0,101,1,Work_Stress,4,3,Tight deadline and late nights for product launch,1.00
1,102,1,Financial_Decision,3,15,Considering switching job for higher salary,0.19
2,103,2,Family_Conflict,2,7,Minor disagreement with spouse about work life...,0.25
3,104,2,Career_Opportunity,4,20,Offered a lateral move to a new brand team,0.19
4,105,3,Health_Concern,5,5,Doctor advised to reduce blood pressure and st...,0.83


In [9]:
user_stress_features = (
    events_df
    .groupby("user_id")
    .agg(
        total_events=("event_id", "count"),
        avg_severity=("severity_score", "mean"),
        max_severity=("severity_score", "max"),
        recent_events=("event_recency_days", lambda x: (x <= 7).sum())
    )
    .fillna(0)
    .reset_index()
)


In [10]:
user_stress_features["high_stress"] = (
    (user_stress_features["avg_severity"] >= 3) |
    (user_stress_features["recent_events"] >= 1)
).astype(int)


In [11]:
from sklearn.model_selection import train_test_split

X = user_stress_features[
    ["total_events", "avg_severity", "max_severity", "recent_events"]
]
y = user_stress_features["high_stress"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3

[[2 0]
 [0 1]]


In [13]:
import numpy as np

stress_features = (
    events_df
    .groupby("user_id")
    .agg(
        total_events=("event_id", "count"),
        avg_severity=("severity_score", "mean"),
        max_severity=("severity_score", "max"),
        recent_events=("event_recency_days", lambda x: (x <= 14).sum())
    )
    .reset_index()
)

stress_features.fillna(0, inplace=True)


In [14]:
stress_features["stress_score"] = (
    0.4 * (stress_features["avg_severity"] / 5) +
    0.3 * (stress_features["max_severity"] / 5) +
    0.2 * (stress_features["recent_events"] / stress_features["total_events"].clip(lower=1)) +
    0.1 * (stress_features["total_events"] / stress_features["total_events"].max())
)

stress_features["stress_score"] = stress_features["stress_score"].clip(0, 1)


In [15]:
def stress_level(score):
    if score >= 0.6:
        return "High"
    elif score >= 0.3:
        return "Medium"
    else:
        return "Low"

stress_features["stress_level"] = stress_features["stress_score"].apply(stress_level)


In [16]:
stress_features[["user_id", "stress_score", "stress_level"]]


,user_id,stress_score,stress_level
0,1,0.3076,Medium
1,2,0.2326,Low
2,3,0.3662,Medium
3,4,0.2400,Low
4,5,0.4838,Medium
5,6,0.0612,Low
6,7,0.2934,Low
7,8,0.3340,Medium
8,9,0.0724,Low
9,10,0.3298,Medium


In [30]:
# Copy to avoid modifying original
stress_df = stress_features.copy()

# Normalize values safely
stress_df["total_events_norm"] = stress_df["total_events"] / stress_df["total_events"].max()
stress_df["avg_severity_norm"] = stress_df["avg_severity"] / stress_df["avg_severity"].max()
stress_df["max_severity_norm"] = stress_df["max_severity"] / stress_df["max_severity"].max()
stress_df["recent_events_norm"] = stress_df["recent_events"] / stress_df["recent_events"].max()

# Final stress score (weighted)
stress_df["stress_score"] = (
    0.25 * stress_df["total_events_norm"] +
    0.30 * stress_df["avg_severity_norm"] +
    0.30 * stress_df["max_severity_norm"] +
    0.15 * stress_df["recent_events_norm"]
)

# Stress level
def stress_level(score):
    if score < 0.3:
        return "Low"
    elif score < 0.6:
        return "Medium"
    else:
        return "High"

stress_df["stress_level"] = stress_df["stress_score"].apply(stress_level)


In [32]:
stress_df[["user_id", "stress_score", "stress_level"]]


,user_id,stress_score,stress_level
0,1,0.686527,High
1,2,0.484431,Medium
2,3,0.573204,Medium
3,4,0.505988,Medium
4,5,0.875000,High
5,6,0.153743,Low
6,7,0.386377,Medium
7,8,0.490569,Medium
8,9,0.182485,Low
9,10,0.479790,Medium


In [33]:
import json

with open("/content/final_rule_engine_output.json", "r") as f:
    users_json = json.load(f)


In [34]:
stress_lookup = {
    int(row["user_id"]): {
        "stress_score": round(float(row["stress_score"]), 4),
        "stress_level": row["stress_level"]
    }
    for _, row in stress_df.iterrows()
}


In [35]:
for user in users_json:
    uid = int(user["user_id"])

    if uid in stress_lookup:
        user["stress_score"] = stress_lookup[uid]["stress_score"]
        user["stress_level"] = stress_lookup[uid]["stress_level"]
    else:
        user["stress_score"] = 0.0
        user["stress_level"] = "Low"


In [36]:
with open("/content/final_rule_engine_output_with_stress.json", "w") as f:
    json.dump(users_json, f, indent=2)
